# War Crimes & Human Rights Violations — Exploratory Data Analysis

**Sources:** ACLED (Armed Conflict Location & Event Data) + HRDAG (Human Rights Data Analysis Group)  
**Data coverage:** Last 5 years globally (ACLED), historical conflict data (HRDAG)  
**Event types:** Battles | Explosions/Remote violence | Violence against civilians | Sexual violence  

---

### Research Questions
1. Who are the actors (state vs. non-state) committing the most documented violations?
2. What event types are most common, and has that changed over time?
3. When and where are violations most concentrated geographically?

---

> **Prerequisite:** Run `python src/ingest.py` before this notebook to populate `data/raw/`.

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap, MarkerCluster
from IPython.display import display, IFrame, HTML

# Add src to path so we can import project modules
sys.path.insert(0, str(Path('..').resolve()))
from src.visualize import (
    load_acled,
    ICC_SITUATION_COUNTRIES,
    _classify_actor_type,
    PALETTE,
    OUT_DIR,
    PROC_DIR,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'

print('Setup complete.')

## 1. Load Data

In [ ]:
# Load ACLED
df = load_acled(RAW_DIR / 'acled_raw.csv')

if df.empty:
    print('\n  No ACLED data found.')
    print('  Run: python src/ingest.py --source acled')
else:
    print(f'ACLED: {len(df):,} rows | {df["country"].nunique()} countries | '
          f'{df["event_date"].min().date()} to {df["event_date"].max().date()}')

In [ ]:
# Load HRDAG (if available)
hrdag_colombia_path = RAW_DIR / 'hrdag_colombia.csv'
hrdag_guatemala_path = RAW_DIR / 'hrdag_guatemala.csv'

df_col = pd.read_csv(hrdag_colombia_path) if hrdag_colombia_path.exists() else pd.DataFrame()
df_guat = pd.read_csv(hrdag_guatemala_path) if hrdag_guatemala_path.exists() else pd.DataFrame()

print(f'HRDAG Colombia:  {len(df_col):,} rows  | columns: {list(df_col.columns[:8]) if not df_col.empty else "(not loaded)"}')
print(f'HRDAG Guatemala: {len(df_guat):,} rows | columns: {list(df_guat.columns[:8]) if not df_guat.empty else "(not loaded)"}')

In [ ]:
# Quick schema overview
if not df.empty:
    df.info()

In [ ]:
# Sample rows
if not df.empty:
    display(df.head(3))

---

## 2. Dataset Overview

In [ ]:
if not df.empty:
    print('=== Event type distribution ===')
    print(df['event_type'].value_counts().to_string())
    print()
    print('=== Top 10 countries ===')
    print(df['country'].value_counts().head(10).to_string())
    print()
    print('=== Year range ===')
    print(df['year'].value_counts().sort_index().to_string())

In [ ]:
# Missing-value audit
if not df.empty:
    missing = df.isnull().sum().sort_values(ascending=False)
    missing_pct = (missing / len(df) * 100).round(2)
    display(
        pd.DataFrame({'missing': missing, 'pct': missing_pct})
        .query('missing > 0')
    )

---

## 3. Research Question 1: Who Are the Actors?

**Key distinction:** ACLED's `inter1` field codes the primary actor type using numeric codes:
- 1 = State Forces  2 = Rebel Groups  3 = Political Militias  4 = Communal Militias
- 5 = Rioters  6 = Protesters  7 = Civilians  8 = External / Other Forces

In [ ]:
if not df.empty and 'inter1' in df.columns:
    df['actor_type'] = df['inter1'].apply(_classify_actor_type)

    actor_counts = df['actor_type'].value_counts().reset_index()
    actor_counts.columns = ['Actor Type', 'Events']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    axes[0].barh(actor_counts['Actor Type'], actor_counts['Events'],
                 color='#c0392b', alpha=0.85)
    axes[0].set_title('Events by Actor Type (Primary Actor)', fontsize=13)
    axes[0].set_xlabel('Event Count')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    # Pie chart
    axes[1].pie(
        actor_counts['Events'],
        labels=actor_counts['Actor Type'],
        autopct='%1.1f%%',
        colors=sns.color_palette('Set2', len(actor_counts)),
        startangle=90,
    )
    axes[1].set_title('Share of Events by Actor Type', fontsize=13)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'actor_type_overview.png', dpi=150)
    plt.show()
    print()
    display(actor_counts)

In [ ]:
# Top 20 named actors
if not df.empty:
    top20 = df['actor1'].value_counts().head(20).sort_values()

    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(top20.index, top20.values, color='#c0392b', alpha=0.85)
    ax.set_title('Top 20 Named Actors by Event Count (ACLED)', fontsize=13)
    ax.set_xlabel('Events')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    for bar, val in zip(bars, top20.values):
        ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'top20_actors.png', dpi=150)
    plt.show()

In [ ]:
# State vs. non-state breakdown by event type
if not df.empty and 'actor_type' in df.columns:
    state_label = lambda x: 'State' if x in ('State Military', 'State Police') else 'Non-State'
    df['state_vs_nonstate'] = df['actor_type'].apply(state_label)

    pivot = (
        df.groupby(['event_type', 'state_vs_nonstate'])
        .size()
        .unstack(fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot(kind='bar', ax=ax, color=['#e74c3c', '#3498db'], edgecolor='white')
    ax.set_title('State vs. Non-State Actors by Event Type', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylabel('Events')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Actor Category')

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'state_vs_nonstate.png', dpi=150)
    plt.show()

    print()
    display(pivot)

---

## 4. Research Question 2: Event Types Over Time

Has the composition of documented violations changed over the 5-year window?

In [ ]:
if not df.empty:
    df['month'] = df['event_date'].dt.to_period('M').dt.to_timestamp()

    monthly = (
        df.groupby(['month', 'event_type'])
        .size()
        .reset_index(name='count')
    )

    fig, ax = plt.subplots(figsize=(14, 5))
    for etype, color in PALETTE.items():
        sub = monthly[monthly['event_type'] == etype]
        ax.plot(sub['month'], sub['count'], label=etype, color=color, linewidth=2)

    ax.set_title('Monthly Events by Type (ACLED)', fontsize=13)
    ax.set_xlabel('Month')
    ax.set_ylabel('Events')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(title='Event Type', loc='upper left')

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'monthly_events_static.png', dpi=150)
    plt.show()

In [ ]:
# Annual totals and year-over-year change
if not df.empty:
    annual = (
        df.groupby(['year', 'event_type'])
        .size()
        .unstack(fill_value=0)
    )
    print('=== Annual event counts by type ===')
    display(annual)

    print('\n=== Year-over-year % change ===')
    display(annual.pct_change().mul(100).round(1))

In [ ]:
# Sexual violence trend specifically — note 2020 methodology change in ACLED
if not df.empty:
    sv = df[df['event_type'] == 'Sexual violence']
    if not sv.empty:
        sv_annual = sv.groupby('year').size()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(sv_annual.index.astype(int), sv_annual.values, color='#2471a3', alpha=0.85)
        ax.set_title('Annual Sexual Violence Events (ACLED)\n'
                     'Note: ACLED added this category in 2020; pre-2020 data is incomplete.',
                     fontsize=12)
        ax.set_xlabel('Year')
        ax.set_ylabel('Events')
        ax.axvline(x=2020, color='red', linestyle='--', linewidth=1, alpha=0.7,
                   label='Category introduced (2020)')
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'sexual_violence_trend.png', dpi=150)
        plt.show()
    else:
        print('No sexual violence events in dataset.')

In [ ]:
# Sub-event type breakdown
if not df.empty and 'sub_event_type' in df.columns:
    sub_counts = df['sub_event_type'].value_counts().head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(sub_counts.index, sub_counts.values, color='#8e44ad', alpha=0.8)
    ax.set_title('Top 20 Sub-Event Types (ACLED)', fontsize=13)
    ax.set_xlabel('Events')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'sub_event_types.png', dpi=150)
    plt.show()

---

## 5. Research Question 3: Geographic Concentration

In [ ]:
# Top 20 countries by event count
if not df.empty:
    top_countries = df['country'].value_counts().head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top_countries.index[::-1], top_countries.values[::-1],
            color='#e67e22', alpha=0.85)
    ax.set_title('Top 20 Countries by Event Count (ACLED)', fontsize=13)
    ax.set_xlabel('Events')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'top20_countries.png', dpi=150)
    plt.show()

In [ ]:
# Plotly choropleth (interactive)
if not df.empty:
    counts = df.groupby('country').size().reset_index(name='event_count')
    fig = px.choropleth(
        counts,
        locations='country',
        locationmode='country names',
        color='event_count',
        color_continuous_scale='OrRd',
        title='Total War-Crimes-Related Events by Country (ACLED)',
        labels={'event_count': 'Events'},
    )
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
    fig.write_html(str(OUT_DIR / 'choropleth_world.html'))
    fig.show()

In [ ]:
# Regional breakdown
if not df.empty and 'region' in df.columns:
    region_year = (
        df.groupby(['year', 'region'])
        .size()
        .reset_index(name='events')
    )

    fig = px.line(
        region_year,
        x='year',
        y='events',
        color='region',
        title='Annual Events by Region (ACLED)',
        labels={'events': 'Events', 'year': 'Year'},
    )
    fig.update_layout(template='plotly_white', legend_title='Region')
    fig.write_html(str(OUT_DIR / 'regional_trends.html'))
    fig.show()

In [ ]:
# Folium event density heatmap
if not df.empty:
    MAX_POINTS = 40_000
    coords = df.dropna(subset=['latitude', 'longitude'])[['latitude', 'longitude']]
    if len(coords) > MAX_POINTS:
        coords = coords.sample(MAX_POINTS, random_state=42)

    m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB dark_matter')
    HeatMap(coords.values.tolist(), radius=8, blur=10, max_zoom=6,
            gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 1.0: 'red'}).add_to(m)

    out_path = str(OUT_DIR / 'heatmap_density.html')
    m.save(out_path)
    print(f'Heatmap saved to: {out_path}')
    display(IFrame(out_path, width='100%', height='500px'))

In [ ]:
# Country + event type heatmap (matrix)
if not df.empty:
    top15 = df['country'].value_counts().head(15).index
    heat_data = (
        df[df['country'].isin(top15)]
        .groupby(['country', 'event_type'])
        .size()
        .unstack(fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(heat_data, annot=True, fmt='d', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Events'})
    ax.set_title('Event Type × Country Matrix (Top 15 Countries)', fontsize=13)
    ax.set_xlabel('Event Type')
    ax.set_ylabel('Country')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'country_event_matrix.png', dpi=150)
    plt.show()

---

## 6. Accountability Gap Analysis

Which high-event countries have **no open ICC situation**?

In [ ]:
if not df.empty:
    country_counts = df['country'].value_counts().head(30).reset_index()
    country_counts.columns = ['country', 'events']
    country_counts['icc_situation'] = country_counts['country'].apply(
        lambda c: 'ICC Open Situation' if c in ICC_SITUATION_COUNTRIES else 'No ICC Situation'
    )

    # Sort: no-ICC countries first to highlight gap
    no_icc = country_counts[country_counts['icc_situation'] == 'No ICC Situation']
    has_icc = country_counts[country_counts['icc_situation'] == 'ICC Open Situation']

    print(f'Top 30 countries by event count:')
    print(f'  With ICC situation: {len(has_icc)}')
    print(f'  Without ICC situation: {len(no_icc)}')
    print()
    print('Countries with high event counts but NO ICC situation:')
    display(no_icc.head(10))

In [ ]:
if not df.empty:
    fig = px.bar(
        country_counts,
        x='country',
        y='events',
        color='icc_situation',
        color_discrete_map={
            'ICC Open Situation': '#2471a3',
            'No ICC Situation': '#e74c3c',
        },
        title='Accountability Gap: ACLED Events vs. ICC Situations (Top 30 Countries)',
        labels={'events': 'Events', 'country': 'Country', 'icc_situation': 'ICC Status'},
    )
    fig.update_layout(
        template='plotly_white',
        xaxis_tickangle=-40,
        legend_title='ICC Status',
    )
    fig.write_html(str(OUT_DIR / 'accountability_gap.html'))
    fig.show()

---

## 7. HRDAG Deep Dive (Colombia)

HRDAG applies multiple systems estimation (MSE) to statistically estimate
**total killings including undocumented ones**. This section explores the Colombia dataset
if it was successfully downloaded.

In [ ]:
if not df_col.empty:
    print('=== HRDAG Colombia: Column overview ===')
    df_col.info()
    display(df_col.head(5))
else:
    print('HRDAG Colombia dataset not available.')
    print('Run: python src/ingest.py --source hrdag')

In [ ]:
# Perpetrator analysis if column exists
if not df_col.empty:
    # Common HRDAG column names — may vary by dataset version
    perp_col = next(
        (c for c in df_col.columns if 'perp' in c.lower() or 'actor' in c.lower() or 'grupo' in c.lower()),
        None
    )
    if perp_col:
        perp_counts = df_col[perp_col].value_counts().head(15)
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(perp_counts.index, perp_counts.values, color='#8e44ad', alpha=0.85)
        ax.set_title(f'HRDAG Colombia: Events by Perpetrator ({perp_col})', fontsize=12)
        ax.set_xlabel('Records')
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'hrdag_colombia_perpetrators.png', dpi=150)
        plt.show()
    else:
        print('No perpetrator column found. Available columns:')
        print(list(df_col.columns))

In [ ]:
# Temporal trend if year column present
if not df_col.empty:
    year_col = next(
        (c for c in df_col.columns if 'year' in c.lower() or 'año' in c.lower() or 'date' in c.lower()),
        None
    )
    if year_col:
        try:
            df_col['_year'] = pd.to_numeric(df_col[year_col], errors='coerce')
            annual_col = df_col.dropna(subset=['_year']).groupby('_year').size()

            fig, ax = plt.subplots(figsize=(12, 4))
            ax.plot(annual_col.index, annual_col.values, marker='o',
                    color='#8e44ad', linewidth=2)
            ax.set_title('HRDAG Colombia: Annual Documented Records', fontsize=12)
            ax.set_xlabel('Year')
            ax.set_ylabel('Records')
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
            plt.tight_layout()
            plt.savefig(OUT_DIR / 'hrdag_colombia_annual.png', dpi=150)
            plt.show()
        except Exception as e:
            print(f'Could not plot year trend: {e}')
    else:
        print('No year column found in HRDAG Colombia data.')

---

## 8. Summary Statistics & Export

In [ ]:
if not df.empty:
    # Global summary
    print('=== Global ACLED Summary ===')
    print(f'Total events:        {len(df):,}')
    print(f'Total countries:     {df["country"].nunique()}')
    print(f'Total fatalities:    {df["fatalities"].sum():,}')
    print(f'Date range:          {df["event_date"].min().date()} to {df["event_date"].max().date()}')
    print()

    # Countries with no ICC situation but high event counts
    high_events_no_icc = (
        df.groupby('country')
        .size()
        .reset_index(name='events')
        .sort_values('events', ascending=False)
        .assign(icc=lambda x: x['country'].isin(ICC_SITUATION_COUNTRIES))
        .query('icc == False')
        .head(10)
    )
    print('=== Top 10 High-Event Countries With No ICC Situation ===')
    display(high_events_no_icc)

In [ ]:
# Export aggregated CSV
if not df.empty:
    from src.visualize import export_summary
    export_summary(df)
    print('Summary CSVs written to data/processed/ and outputs/')

In [ ]:
# Generate all static + interactive outputs (calls visualize.py functions)
# Remove '#' below to run all charts from the notebook

# from src.visualize import (
#     chart_choropleth_world,
#     chart_event_cluster_map,
#     chart_heatmap_density,
#     chart_monthly_events_by_type,
#     chart_animated_timeseries,
#     chart_yoy_violence_civilians,
#     chart_top20_actors,
#     chart_actor_type_by_region,
#     chart_actor_network,
#     chart_accountability_gap,
#     chart_data_completeness,
# )
#
# for fn in [chart_choropleth_world, chart_event_cluster_map, chart_heatmap_density,
#            chart_monthly_events_by_type, chart_animated_timeseries,
#            chart_yoy_violence_civilians, chart_top20_actors,
#            chart_actor_type_by_region, chart_actor_network,
#            chart_accountability_gap, chart_data_completeness]:
#     fn(df)

print('Notebook complete. To regenerate all outputs: python src/visualize.py')

---

## 9. Findings Summary

*(Fill in after running with your actual data — these are placeholders.)*

### Key Findings

**Actors**
- Non-state armed groups (rebel groups, militias) account for the plurality of documented events.
- State military actors appear disproportionately in [region] compared to others.
- Top named actors by event count: see `outputs/top20_actors.png`.

**Event Types Over Time**
- Battles consistently represent the largest share of events.
- Violence against civilians has [trended up/down] since [year].
- Sexual violence data shows a sharp increase after 2020 — likely a reporting artifact
  from ACLED's methodology change, not a real-world increase.

**Geography**
- The highest event concentrations are in [countries].
- [Region] has seen the largest year-over-year increase.
- Event density clusters in [geographic areas].

**Accountability Gap**
- Several high-event countries have no open ICC situation, including [list countries].
- This reflects ICC jurisdiction limitations, Security Council veto dynamics, and
  non-ratification of the Rome Statute.

---

### Methodological Caveats

1. ACLED counts **events**, not individual victims. Event counts are not a proxy for severity.
2. The sexual violence category was introduced in 2020; pre-2020 comparisons are unreliable.
3. Countries with restricted press access are systematically undercounted.
4. HRDAG data covers historical conflicts and should not be compared directly to ACLED's
   near-real-time global data without methodological adjustment.
5. ICC situation data is as of 2024 and subject to change.